# 02. Growing window

**Tier:** Deep. **Why you'd use this:** Grow a window over the string. Remember the last index of each character.

**After this lesson you can:** grow and shrink a window to meet a condition; track the shortest length that works.
About 20 minutes. **Uses:** [Fixed window](01-fixed-window-lesson.ipynb)


## Quick review

1. When do you grow the right edge of a growing window?
2. When do you shrink the left edge?

<details><summary>Answers</summary>

For 02 shortest window lesson: skim [Fixed window](01-fixed-window-lesson.ipynb) if needed.
</details>


## Recognize it

You need the shortest contiguous span whose sum reaches a target (or another monotone condition). The prompt says "smallest length", "minimum subarray", or "grow until valid, then shrink".

Signals: contiguous values, a threshold, and length as the answer. A look-alike that is not this pattern: longest substring with unique characters — still a variable window, but the shrink rule is about repeats, not about a numeric total. Another look-alike: fixed length `k` — use the fixed-window lesson.

If values can be negative, say so out loud before coding: the grow-and-shrink proof needs monotonic sums, and negatives break that.


## Why it works this way

When every value is positive, growing the right edge only raises the window sum, and shrinking the left edge only lowers it. That monotonicity lets one left index walk forward forever. Expand `right` across the list. Whenever the sum is at least the target, record the length, then advance `left` while the sum still qualifies. Each index enters the window once and leaves at most once, so time stays proportional to `n`. Extra memory is constant.

A nested approach tries every left edge and grows a right edge from there, restarting the sum each time. That costs about `n²` additions on dense inputs. Prefix sums plus binary search can also answer shortest-sum queries when you precompute totals, but they spend extra memory and logarithmic probes for each left edge. The two-pointer window is the linear special case for non-negative values, and it is usually the first tool to reach for in timed challenges when the constraint matches.

If negatives are allowed, shrinking is no longer safe: dropping a negative would raise the sum, so the "while valid, shrink" loop can skip the true shortest span. Then you need a different tool — prefix sums with an ordered structure, or a constrained search that does not assume monotonicity. State that limit when you name the pattern so you do not force a variable window onto the wrong input.

The invariant during the scan: `total` is the sum of `nums[left : right + 1]`, and `left` never passes `right + 1`. Returning `0` when no window works is the usual empty answer for this sibling. Seeding `best` with `len(nums) + 1` (or infinity) makes "no window found" easy to detect after the scan without a separate boolean flag.

The design reason for one left pointer that never retreats is the same monotonicity: once a left edge is too heavy to keep, every later right edge will still see that left value as optional dead weight for a shortest span. Moving left forward permanently is therefore safe, and that permanence is what keeps the algorithm linear.


## When to use it

Use a grow-and-shrink window when values are non-negative and you want the shortest (or longest) span that meets a sum or count threshold. Use a fixed window when length is given. Use prefix sums when values may be negative or ranges are arbitrary. Use a hash map window when the constraint is about distinct characters rather than a numeric total.

Prefer prefix-plus-binary-search when you already built prefixes for other queries and only need a few shortest-sum answers. Prefer the linear window when this question stands alone and values stay non-negative.


## Brute force, and why it's slow

For every left edge, grow right until the sum hits the target. Count the additions.


In [1]:
def min_subarray_len_slow(target, nums):
    additions = 0
    best = len(nums) + 1
    for left in range(len(nums)):
        total = 0
        for right in range(left, len(nums)):
            total += nums[right]
            additions += 1
            if total >= target:
                best = min(best, right - left + 1)
                break
    return (0 if best > len(nums) else best), additions


print(min_subarray_len_slow(7, [2, 3, 1, 2, 4, 3]))
print(min_subarray_len_slow(100, [1, 2, 3]))


(2, 17)
(0, 6)


## Key insight

With non-negative values, one expanding right edge and one never-backward left edge visit each index a constant number of times while hunting the shortest valid span. Shrink only while the window still meets the target.


## Worked example

Return the shortest subarray whose sum is at least `target`, or `0` when none exists.


In [2]:
def min_subarray_len(target: int, nums: list[int]) -> int:
    # 1. best starts longer than any real window.
    best = len(nums) + 1
    total = 0
    left = 0
    for right, value in enumerate(nums):
        # 2. Grow the right edge.
        total += value
        # 3. Shrink from the left while the window still qualifies.
        while total >= target and left <= right:
            best = min(best, right - left + 1)
            total -= nums[left]
            left += 1
    return 0 if best > len(nums) else best


print(min_subarray_len(7, [2, 3, 1, 2, 4, 3]))
print(min_subarray_len(4, [1, 4, 4]))
print(min_subarray_len(11, [1, 1, 1, 1]))


2
1
0


## Trace

`target = 7`, nums `[2, 3, 1, 2, 4, 3]`:

| right | total after grow | shrink actions | best |
|---|---|---|---|
| 0 | 2 | — | ∞ |
| 1 | 5 | — | ∞ |
| 2 | 6 | — | ∞ |
| 3 | 8 | len 4, drop 2 → 6 | 4 |
| 4 | 10 | len 4, drop 3 → 7; len 3, drop 1 → 6 | **3** |
| 5 | 9 | len 3, drop 2 → 7; len 2, drop 4 → 3 | **2** |


## Cost

Each index is added once and subtracted at most once, so time grows with `n`. Extra memory is a few integers. The nested restart pays about `n² / 2` additions on hard inputs. Prefix-plus-binary-search trades extra memory and log factors for similar answers when sums are monotone.


## Variants

Not solved here: [Minimum Window Substring](../../challenges/problems/41-minimum-window-substring.ipynb), [Longest Substring Without Repeating Characters](../../challenges/problems/03-longest-substring-without-repeating-characters.ipynb), [Longest Repeating Character Replacement](../../challenges/problems/45-longest-repeating-character-replacement.ipynb).


## Common mistake

This bug is silent. The shrink loop uses `if` instead of `while`, so after one shrink the sum may still be valid and a shorter length is missed. There is no traceback. The print is `3`, and the answer is `2`.


In [3]:
def min_subarray_len_wrong(target, nums):
    best = len(nums) + 1
    total = 0
    left = 0
    for right, value in enumerate(nums):
        total += value
        if total >= target and left <= right:  # shrinks only once
            best = min(best, right - left + 1)
            total -= nums[left]
            left += 1
    return 0 if best > len(nums) else best


print(min_subarray_len_wrong(7, [2, 3, 1, 2, 4, 3]))


4


The fix keeps shrinking with `while` until the window fails the target. The buggy cell prints `4` for target `7` on `[2, 3, 1, 2, 4, 3]` because it shrinks only once per step.


In [4]:
print(min_subarray_len(7, [2, 3, 1, 2, 4, 3]))


2


## Python idioms

`enumerate` supplies the right edge. A sentinel `best = len(nums) + 1` avoids a separate "found" flag when you convert "still impossible" into `0` at the end.


In [5]:
def min_subarray_len_copy(target: int, nums: list[int]) -> int:
    best = float("inf")
    total = left = 0
    for right, value in enumerate(nums):
        total += value
        while total >= target:
            best = min(best, right - left + 1)
            total -= nums[left]
            left += 1
    return 0 if best == float("inf") else best


print(min_subarray_len_copy(7, [2, 3, 1, 2, 4, 3]))


2


## Recap

- Grow the right edge to meet the target.
- Shrink the left edge while the window still qualifies.
- Non-negative values make the one-pass shrink safe.

Next: [practice](02-shortest-window-practice.ipynb).
